# 01 · Environment inspection and setup
Inspect the SageMaker environment **before** installing anything, then install only what is missing.
Nothing here creates AWS resources. Expected hardware: `ml.g6.xlarge` (NVIDIA L4, 24 GB) or `ml.g5.xlarge` (A10G, 24 GB).

In [ ]:
# Run from anywhere: find the project root and make it importable
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
PY = sys.executable
print("project root:", ROOT)

## 1. Raw diagnostics

In [ ]:
!nvidia-smi

In [ ]:
!{PY} --version
!{PY} -m pip show torch transformers accelerate 2>/dev/null | grep -E '^(Name|Version)'
!df -h

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)

        print("GPU:", p.name)
        print("Total VRAM:", total / 1024**3)
        print("Free VRAM:", free / 1024**3)

## 2. Install missing packages (does not replace a working CUDA PyTorch) and write `requirements-lock.txt`

In [ ]:
!bash setup_env.sh

## 3. Hugging Face token and cache location

In [ ]:
# Hugging Face token: kept in this kernel's environment only, never written to disk or printed.
# Qwen2.5-Coder is not gated, so the token is optional, but it avoids anonymous rate limits.
import os, getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (Enter to skip): ").strip()
    if not os.environ["HF_TOKEN"]:
        del os.environ["HF_TOKEN"]
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
# Put the model cache on the persistent volume of the space
os.environ.setdefault("HF_HOME", str(Path.home() / "hf_cache"))
print("HF_HOME:", os.environ["HF_HOME"])

## 4. Full server report → `results/server_report.json`

In [ ]:
!{PY} scripts/server_report.py

In [ ]:
import json
rep = json.load(open("results/server_report.json"))
{k: rep[k] for k in ("sagemaker_environment", "aws_region", "instance_type", "python")}

## 5. Can Qwen2.5-Coder-7B-Instruct run here at 8K? (estimate; notebook 02 verifies)

In [ ]:
!{PY} scripts/check_gpu.py --model Qwen/Qwen2.5-Coder-7B-Instruct --context 8192

## 6. Unit tests that need no GPU model download (evaluation, results, config)

In [ ]:
!{PY} -m pytest tests/test_gpu.py tests/test_evaluation.py tests/test_results.py -q